# TSFS12 Hand-in exercise 4: Collaborative Control 5.1

In [52]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [53]:
%matplotlib qt

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [ ]:
def single_integrator(t, x, u, model_params):
    '''
    System-model functions
     
     Input arguments are:
     t - time 
     x - agent state
     u - the control input u, 
     model_params - structure with parameters of the model '''

    return u

def controller_p_v(y, y_ref, ctrl_params):
    # Absolute position controller for top and bottom agents
    Kp = ctrl_params["kp"]
    return Kp * (y_ref - y)


def controller_relative(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    u = np.zeros(2)

    for relative_position, desired_distance in zip(y, y_ref):
        distance = np.linalg.norm(relative_position)
        u += 4 * Kp * (distance**2 - desired_distance**2) * relative_position

    return u


Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [55]:
n = 2  # (px, py)
G = [ #(i, j) means agent j measures agent i
    (0,),

    # Minimally rigid graph
    (0, 1), (0, 2),
    (1, 2), (1, 3),
    (2, 3), (2, 4),
    (3, 4), (3, 5),
    (4, 5), (4, 6),
    (5, 6),

    (2, 1), (3, 1),
    (3, 2), (4, 2),
    (4, 3), (5, 3),
    (5, 4), (6, 4),
    (6, 5), (7, 5),
    (7, 6),

    (7,)
]

desired_positions = np.array([
    [0, 6], [-1, 5], [1, 5], [-1, 3],
    [1, 3], [-1, 1], [1, 1], [0, 0]
])

def distance_reference(agent_idx):
    neighbors = [edge[0] for edge in G if len(edge) == 2 and edge[1] == agent_idx]
    return np.array([
        np.linalg.norm(desired_positions[neighbor] - desired_positions[agent_idx])
        for neighbor in neighbors
    ])

formation_references = [
    # Top agent: circle of radius 2 centered at its desired location (0, 6)
    lambda t: np.array([2 * np.cos(t), 6 + 2 * np.sin(t)]),
    # Other agents: desired distances to neighbors
    lambda t, i=1: distance_reference(i),
    lambda t, i=2: distance_reference(i),
    lambda t, i=3: distance_reference(i),
    lambda t, i=4: distance_reference(i),
    lambda t, i=5: distance_reference(i),
    lambda t, i=6: distance_reference(i),

    # Bottom agent also has absolute positioning
    lambda t: np.array([2 * np.cos(t), 0 + 2 * np.sin(t)])
]


Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [ ]:
# Controller parameters
ctrl_params = {
    "kp": 1.0,
}

agents = []

# Agent 0 has absolute measurement
agents.append(
    CreateAgent(
        single_integrator,
        {},
        controller_p_v,
        ctrl_params
    )
)

# Agents 1-6 use relative measurements
for _ in range(6):
    agents.append(
        CreateAgent(
            single_integrator,
            {},
            controller_relative,
            ctrl_params
        )
    )

# Agent 7 has absolute measurement
agents.append(
    CreateAgent(
        single_integrator,
        {},
        controller_p_v,
        ctrl_params
    )
)


formation = AgentFormation(
    agents,
    G,
    formation_references,
    n
)

# Start in the reference shape, translated so agent 0 begins at its
# circular reference at t=0: (2, 6).
x0 = (desired_positions + np.array([2.0, 0.0])).reshape(-1)

Simulate the formation

In [57]:
t = np.arange(0, 20, 0.05) # Time vector for the simulation
x = odeint(formation.ode, x0, t)

Animate the results

In [ ]:
fig, (ax, ax_dist) = plt.subplots(
    1, 2, num=20, clear=True, figsize=(12, 6),
    gridspec_kw={"width_ratios": [1.2, 1]}
)
fig.subplots_adjust(left=0.06, right=0.98, wspace=0.28)
ax.axis([-6, 8, -4, 10])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Distance-based formation control (Kp = {ctrl_params['kp']})")

# Distance from the top node (agent 0) to the first node below it (agent 1).
distance_top_first = np.linalg.norm(x[:, 2:4] - x[:, 0:2], axis=1)
target_distance = np.linalg.norm(desired_positions[1] - desired_positions[0])
ax_dist.plot(t, distance_top_first, color="tab:blue", label="Measured distance")
ax_dist.axis([0, 20, 0, 2.5])
ax_dist.axhline(target_distance, color="tab:red", linestyle="--", label="Target")
ax_dist.set_xlabel("Time [s]")
ax_dist.set_ylabel("Distance")
ax_dist.set_title("Distance between top agent and first agent")
ax_dist.grid(True)
ax_dist.legend()

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:cyan", "tab:purple", "tab:brown", "tab:pink"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x[: i + 1, n * idx]
        py = x[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text,)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x.shape[0], blit=True, repeat=False
)
xf = x[-1].reshape(8, 2)

for i in range(8):
    print(i, xf[i])

0 [1.34460901 6.43820849]
1 [0.38566431 5.17070291]
2 [2.31511275 5.15889301]
3 [0.34432176 3.12896585]
4 [2.34323148 3.11515119]
5 [0.33257707 1.12633401]
6 [2.391595 1.11283 ]
7 [1.34460901 0.43820849]


: 